Step 1: Understanding A/B Testing;

An A/B test splits users into two groups:
- *Control Group (A)*: Sees the original version
- *Treatment Group (B)*: Sees the new version

Step 2: Generate Sample Data

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

np.random.seed(42)

# A/B Test data

n_users = 10000
# Control group (A) - 10% conversion rate

control_size = 5000
control_conversion = np.random.binomial(1, 0.10, control_size)

# Treatment group (B) - 12% conversion rate (20% relative improvement)
treatment_size = 5000
treatment_conversion = np.random.binomial(1, 0.12, treatment_size)

# Create DataFrame
data = pd.DataFrame({
    'user_id': range(n_users),
    'group': ['control'] * control_size + ['treatment'] * treatment_size,
    'converted': np.concatenate([control_conversion, treatment_conversion])
})


Step 3: Basic Data Exploration

In [2]:
summary = data.groupby('group').agg({
    'converted': ['count', 'sum', 'mean']
}).round(4)
print(summary)

          converted             
              count  sum    mean
group                           
control        5000  479  0.0958
treatment      5000  567  0.1134


Step 4: Calculate Key Metrics

In [3]:
# Extract metrics for each group
control_converted = data[data['group'] == 'control']['converted'].sum()
control_total = data[data['group'] == 'control']['converted'].count()
control_total != 0
control_rate = control_converted / control_total

treatment_converted = data[data['group'] == 'treatment']['converted'].sum()
treatment_total = data[data['group'] == 'treament']['converted'].count()
treatment_total != 0
treatment_rate = treatment_converted / treatment_total


C:\Users\Aqib\AppData\Local\Temp\ipykernel_3520\1994288127.py:10: RuntimeWarning: divide by zero encountered in scalar divide
  treatment_rate = treatment_converted / treatment_total


In [4]:
# Calculate absolute and relative lift
absolute_lift = treatment_rate - control_rate
relative_lift = (treatment_rate - control_rate) / control_rate * 100

In [5]:
# Printing All Values
print(f'Control Rate: {control_rate:.2%}')
print(f'Treatment Rate: {treatment_rate:.2%}')
print(f'Absolute Lift: {absolute_lift:.2%}')
print(f'Relarive Lift: {relative_lift:.1%}')


Control Rate: 9.58%
Treatment Rate: inf%
Absolute Lift: inf%
Relarive Lift: inf%


Step 5: Statistical Significance Testing
This is the core - determining if the difference is real or just random chance.

In [6]:
# Two-proportion z-test
from statsmodels.stats.proportion import proportions_ztest

# Control of Conversios
conversions = np.array([treatment_converted, control_converted])

# Total users in each conversions
total = np.array([treatment_converted,control_converted])


In [7]:
# Performing z_test
z_test, p_value = proportions_ztest(conversions, total)

# printing z_test, p_test values
print(f'Z_Test: {z_test:.4f}')
print(f'P_Test: {p_value:.4f}')
print(f'Significance of 90% conversions: {p_value<0.05}')


Z_Test: nan
P_Test: nan
Significance of 90% conversions: False


c:\Users\Aqib\AppData\Local\Programs\Python\Python314\Lib\site-packages\statsmodels\stats\weightstats.py:792: RuntimeWarning: invalid value encountered in scalar divide
  zstat = value / std


Step 6: Confidence Intervals

In [8]:
# Calculate 95% confidence interval for the difference
def confidence_interval_proportion(success, total, confidence=0.95):
    prop = success / total
    z_score = stats.norm.ppf((1+confidence) / 2)
    seq = np.sqrt(prop * (1-prop) / total)
    margin = seq * z_score
    return prop - margin, prop + margin


In [9]:
control_ci = confidence_interval_proportion(control_converted, control_total)
treatment_ci = confidence_interval_proportion(treatment_converted, treatment_total)

C:\Users\Aqib\AppData\Local\Temp\ipykernel_3520\839542817.py:3: RuntimeWarning: divide by zero encountered in scalar divide
  prop = success / total
C:\Users\Aqib\AppData\Local\Temp\ipykernel_3520\839542817.py:5: RuntimeWarning: invalid value encountered in sqrt
  seq = np.sqrt(prop * (1-prop) / total)


In [10]:
print(f'Control 95% CI: [{control_ci[0]:.2%}, {control_ci[1]:.25}]')
print(F'Treatment 95% CI: [{treatment_ci[0]:.2%}, {treatment_ci[1]:.2%}]')

Control 95% CI: [8.76%, 0.1039578946903149114788789]
Treatment 95% CI: [nan%, nan%]


In [ ]:
# Conversion Rate Comparison
plt.figure(figsize=(10, 6))

groups = ['Control', 'Treatment']
rates = [control_rate, treatment_rate]
colors = ['#3498db', '#2ecc71']

bars = plt.bar(groups, rates, color=colors, alpha=0.7, edgecolor='black')

# Add error bars (confidence intervals)
ci_ranges = [
    control_ci[1] - control_rate,
    treatment_ci[1] - treatment_rate
]
plt.errorbar(groups, rates, yerr=ci_ranges, fmt='none', 
             color='black', capsize=10, capthick=2)

# Add value labels on bars
for bar, rate in zip(bars, rates):
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height,
             f'{rate:.2%}',
             ha='center', va='bottom', fontsize=12, fontweight='bold')

plt.ylabel('Conversion Rate', fontsize=12)
plt.title('A/B Test: Conversion Rate Comparison', fontsize=14, fontweight='bold')
plt.ylim(0, max(rates) * 1.3)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 5))

# Simulate sampling distributions using bootstrapping
n_bootstrap = 10000

control_bootstrap = np.random.binomial(control_total, control_rate, n_bootstrap) / control_total
treatment_bootstrap = np.random.binomial(treatment_total, treatment_rate, n_bootstrap) / treatment_total

plt.subplot(1, 2, 1)
plt.hist(control_bootstrap, bins=50, alpha=0.7, color='#3498db', 
         label='Control', density=True)
plt.hist(treatment_bootstrap, bins=50, alpha=0.7, color='#2ecc71', 
         label='Treatment', density=True)
plt.xlabel('Conversion Rate')
plt.ylabel('Density')
plt.title('Sampling Distribution of Conversion Rates')
plt.legend()
plt.grid(alpha=0.3)

# Difference distribution
plt.subplot(1, 2, 2)
difference = treatment_bootstrap - control_bootstrap
plt.hist(difference, bins=50, color='purple', alpha=0.7, edgecolor='black')
plt.axvline(0, color='red', linestyle='--', linewidth=2, label='No difference')
plt.axvline(absolute_lift, color='green', linestyle='-', linewidth=2, label='Observed lift')
plt.xlabel('Difference in Conversion Rate')
plt.ylabel('Frequency')
plt.title('Distribution of Treatment Effect')
plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()


In [14]:
from statsmodels.stats.power import zt_ind_solve_power

effect_size = absolute_lift / np.sqrt(control_rate * (1 - control_rate))
power = zt_ind_solve_power(effect_size=effect_size, 
                           nobs1=control_total, 
                           alpha=0.05, 
                           ratio=1.0)

print(f"Statistical Power: {power:.2%}")

Statistical Power: 100.00%


In [15]:
def ab_test_report(data, control_name='control', treatment_name='treatment'):
    """Complete A/B test analysis"""
    
    # Calculate metrics
    control_data = data[data['group'] == control_name]['converted']
    treatment_data = data[data['group'] == treatment_name]['converted']
    
    control_conv = control_data.sum()
    control_total = len(control_data)
    control_rate = control_conv / control_total
    
    treatment_conv = treatment_data.sum()
    treatment_total = len(treatment_data)
    treatment_rate = treatment_conv / treatment_total
    
    # Statistical test
    conversions = np.array([treatment_conv, control_conv])
    totals = np.array([treatment_total, control_total])
    z_stat, p_value = proportions_ztest(conversions, totals, alternative='larger')
    
    


In [ ]:

# Report
print("="*50)
print("A/B TEST RESULTS")
print("="*50)
print(f"\nControl Group:")
print(f"  Users: {control_total:,}")
print(f"  Conversions: {control_conv:,}")
print(f"  Rate: {control_rate:.2%}")
    
print(f"\nTreatment Group:")
print(f"  Users: {treatment_total:,}")
print(f"  Conversions: {treatment_conv:,}")
print(f"  Rate: {treatment_rate:.2%}")
    
print(f"\nLift:")
print(f"  Absolute: {(treatment_rate - control_rate):.2%}")
print(f"  Relative: {((treatment_rate - control_rate) / control_rate * 100):.1f}%")
    
print(f"\nStatistical Significance:")
print(f"  Z-statistic: {z_stat:.4f}")
print(f"  P-value: {p_value:.4f}")
print(f"  Significant (α=0.05): {'YES ✓' if p_value < 0.05 else 'NO ✗'}")
    

In [ ]:
if p_value < 0.05:
        print(f"\n The treatment shows statistically significant improvement!")
else:
    print(f"\n No significant difference detected. Don't implement the change.")

ab_test_report(data)